<a href="https://colab.research.google.com/github/flathfk/Bootcamp_Study/blob/main/260319_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 충정일보 신문 플랫폼 개발(2H)

### 1. 데이터베이스 구성 (MariaDB)

- **테이블명**: `news`
- **컬럼 구성**:
    - `id`: 고유 번호 (기본키, 자동 증가)
    - `title`: 기사 제목
    - `category`: 카테고리 (정치, 경제, 연예 등)
    - `content`: 기사 본문
    - `author`: 기자 이름

### 2. 핵심 기능 정의

### **① 기사 목록 조회 (GET /api/news)**

- **설명**: MariaDB의 `news` 테이블에서 모든 기사 데이터를 가져옵니다.
- **동작**:
    1. `SELECT` 쿼리를 실행하여 `id`, `title`, `category`, `author` 등을 가져옴.
    2. 최신 기사가 먼저 나오도록 `ORDER BY id DESC` 적용.
    3. `await pool.query(sql)`를 통해 결과가 나올 때까지 대기 후 JSON 응답.

### **② 기사 송고/등록 (POST /api/news)**

- **설명**: 기자가 입력한 데이터를 DB에 새 레코드로 삽입합니다.
- **동작**:
    1. `req.body`에서 제목, 카테고리, 본문, 기자명을 전달받음.
    2. `INSERT INTO news (...) VALUES (?, ?, ...)` 쿼리 실행.
    3. `await pool.query(sql, [데이터])`를 통해 저장이 완료될 때까지 대기 후 성공 메시지 응답.

### 3. 클라이언트 기능 요건 (HTML/JS)

### **① 화면 구성 (Grid Layout)**

- CSS Grid 레이아웃 적용 요건
    
    **1. 데스크톱 중심의 다단(Multi-column) 그리드 설계**
    
    - **신문 지면 구조**: 부모 컨테이너에 `display: grid`를 적용하고, 전체 화면을 **12개 이상의 세밀한 컬럼**으로 나누어 실제 종이 신문과 같은 유연한 배치를 구현합니다.
    - **고정 폭 레이아웃**: 모바일을 고려하지 않으므로, 최적의 가독성을 위해 중앙 집중형의 **넓은 고정 폭(예: 1200px 이상)** 컨테이너를 기준으로 설계합니다.
    
    **2. 기사 중요도에 따른 영역 배분 (Grid-Span)**
    
    - **헤드라인 강조**: 가장 중요한 최신 기사는 `grid-column: span 8;` 등을 사용하여 화면의 넓은 면적을 차지하게 하고, 일반 기사는 남은 공간에 배치하여 시각적 위계질서를 잡습니다.
    - **뉴스 섹션 구분**: 좌측에는 메인 기사(Main Content), 우측에는 사이드바(Trend/Rank) 형태의 구성을 `grid-template-areas`를 통해 명확히 분리합니다.
    
    **3. 카드 UI 및 간격 제어**
    
    - **신문 특유의 여백**: `column-gap`과 `row-gap`을 활용하여 기사 간의 경계를 명확히 하고, 실제 신문의 '단'과 '줄' 사이의 여백 느낌을 재현합니다.
    - **이미지 및 텍스트 정렬**: 각 뉴스 카드는 그리드 셀 안에서 이미지와 텍스트가 조화롭게 배치되도록 내부 요소에도 그리드 또는 정렬 속성을 적용합니다.

### **② 비동기 데이터 처리 (Async/Await)**

- **조회**: 페이지 로드 시 `await fetch("/api/news")`를 호출하여 기사를 화면에 그림.
- **송고**: 폼 제출 시 `await fetch("/api/news", { method: 'POST' ... })`를 실행하여 DB에 저장한 후, 목록을 즉시 갱신함.



---



In [54]:
%%bash
pkill -f "node server.js"
pkill -f "cloudflared"
sleep 1
echo "done"

done




---



In [56]:
%%bash
apt install -y mariadb-server mariadb-client -q
service mariadb start
mysql -u root -e "
CREATE DATABASE IF NOT EXISTS testdb;
CREATE USER IF NOT EXISTS 'testuser'@'localhost' IDENTIFIED BY '1234';
GRANT ALL PRIVILEGES ON testdb.* TO 'testuser'@'localhost';
FLUSH PRIVILEGES;
SELECT 1;
"

Reading package lists...
Building dependency tree...
Reading state information...
mariadb-client is already the newest version (1:10.6.23-0ubuntu0.22.04.1).
mariadb-server is already the newest version (1:10.6.23-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.
 * Starting MariaDB database server mariadbd
   ...done.
1
1




---



In [57]:
%%bash
mkdir -p /content/chungjung/static /content/chungjung/templates



---



In [58]:
%%writefile /content/chungjung/package.json

{
  "name": "chungjung-ilbo",
  "main": "server.js",
  "dependencies": {
    "express": "^4.18.2",
    "mysql2": "^3.6.0"
  }
}

Overwriting /content/chungjung/package.json


In [93]:
%%writefile /content/chungjung/static/style.css

* { box-sizing: border-box; margin: 0; padding: 0; }

body {
    background: #ffffff;
    color: #1a1a1a;
    font-family: 'Apple SD Gothic Neo', 'Noto Sans KR', 'Malgun Gothic', Arial, sans-serif;
    line-height: 1.6;
    font-size: 14px;
}

a { color: inherit; text-decoration: none; }

.newspaper-header {
    background: #fff;
    border-bottom: 3px solid #005c2e;
}

.header-inner {
    max-width: 1280px;
    margin: 0 auto;
    padding: 0 24px;
}

.header-logo {
    display: flex;
    justify-content: center;
    padding: 18px 0 14px;
    border-bottom: 1px solid #eee;
}

.header-top {
    display: flex;
    justify-content: flex-end;
    padding: 6px 0;
    border-bottom: 1px solid #eee;
}

.header-meta-right {
    font-size: 12px;
    color: #999;
    text-align: right;
    line-height: 1.8;
}

.header-nav {
    display: flex;
    padding: 0;
}

.header-nav a {
    padding: 10px 18px;
    font-size: 14px;
    font-weight: 600;
    color: #333;
    border-right: 1px solid #eee;
    transition: color 0.15s;
}

.header-nav a:first-child { border-left: 1px solid #eee; }
.header-nav a:hover, .header-nav a.active { color: #005c2e; }

.page {
    max-width: 1280px;
    margin: 0 auto;
    padding: 24px 24px 60px;
}

.section-header {
    display: flex;
    align-items: center;
    gap: 8px;
    margin-bottom: 16px;
    padding-bottom: 10px;
    border-bottom: 2px solid #1a1a1a;
}

.section-label { font-size: 16px; font-weight: 800; color: #1a1a1a; }
.section-date { font-size: 12px; color: #999; margin-left: auto; }

.news-grid {
    display: grid;
    grid-template-columns: repeat(12, 1fr);
    border: 1px solid #e8e8e8;
}

.card-headline {
    grid-column: span 8;
    border-right: 1px solid #e8e8e8;
    padding: 24px 28px;
    background: #fff;
    cursor: pointer;
    transition: background 0.1s;
}
.card-headline:hover { background: #fafafa; }

.news-sidebar {
    grid-column: span 4;
    display: flex;
    flex-direction: column;
}

.card-side {
    flex: 1;
    padding: 16px 20px;
    background: #fff;
    border-bottom: 1px solid #e8e8e8;
    cursor: pointer;
    transition: background 0.1s;
}
.card-side:last-child { border-bottom: none; }
.card-side:hover { background: #fafafa; }

.news-grid-bottom {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    border: 1px solid #e8e8e8;
    border-top: none;
}

.card-normal {
    padding: 18px 22px;
    background: #fff;
    border-right: 1px solid #e8e8e8;
    cursor: pointer;
    transition: background 0.1s;
}
.card-normal:last-child { border-right: none; }
.card-normal:hover { background: #fafafa; }

.card-category {
    display: inline-block;
    font-size: 11px;
    font-weight: 700;
    color: #005c2e;
    margin-bottom: 8px;
    letter-spacing: 0.5px;
}

.card-title {
    font-size: 20px;
    font-weight: 700;
    line-height: 1.4;
    margin-bottom: 10px;
    color: #1a1a1a;
    letter-spacing: -0.3px;
}

.card-headline .card-title { font-size: 26px; letter-spacing: -0.5px; }
.card-side .card-title { font-size: 15px; }
.card-normal .card-title { font-size: 16px; }

.card-content {
    font-size: 13px;
    color: #666;
    line-height: 1.75;
    display: -webkit-box;
    -webkit-line-clamp: 3;
    -webkit-box-orient: vertical;
    overflow: hidden;
}

.card-headline .card-content { font-size: 14px; color: #444; -webkit-line-clamp: 5; }
.card-side .card-content { font-size: 12px; -webkit-line-clamp: 2; }

.card-meta {
    font-size: 11px;
    color: #aaa;
    margin-top: 12px;
    display: flex;
    justify-content: space-between;
    padding-top: 10px;
    border-top: 1px solid #f0f0f0;
}

.card-divider {
    border: none;
    border-top: 2px solid #005c2e;
    margin: 10px 0 14px;
    width: 36px;
}

.empty-state {
    grid-column: 1 / -1;
    text-align: center;
    padding: 80px 20px;
    color: #bbb;
    font-size: 14px;
    background: #fafafa;
}

.form-page {
    max-width: 720px;
    margin: 0 auto;
    padding: 32px 24px 60px;
}

.form-header {
    margin-bottom: 24px;
    padding-bottom: 14px;
    border-bottom: 2px solid #1a1a1a;
}

.form-header h1 {
    font-size: 24px;
    font-weight: 800;
    letter-spacing: -0.5px;
}

.form-header p { font-size: 13px; color: #999; margin-top: 6px; }

.form-panel {
    background: #fff;
    border: 1px solid #e8e8e8;
    padding: 28px;
}

.form-group { margin-bottom: 18px; }

label {
    display: block;
    font-size: 12px;
    font-weight: 700;
    color: #555;
    margin-bottom: 7px;
}

input[type="text"], select, textarea {
    width: 100%;
    border: 1px solid #ddd;
    border-radius: 3px;
    padding: 10px 12px;
    font-size: 14px;
    font-family: 'Apple SD Gothic Neo', 'Noto Sans KR', sans-serif;
    color: #1a1a1a;
    background: #fff;
    outline: none;
    transition: border-color 0.15s;
}

input[type="text"]:focus, select:focus, textarea:focus { border-color: #005c2e; }

textarea { min-height: 200px; resize: vertical; line-height: 1.8; }

.button-row { display: flex; gap: 8px; margin-top: 20px; }

.btn {
    flex: 1;
    padding: 11px;
    border: none;
    cursor: pointer;
    font-size: 14px;
    font-weight: 700;
    font-family: inherit;
    border-radius: 3px;
    transition: opacity 0.15s;
}
.btn:hover { opacity: 0.85; }
.btn-primary { background: #005c2e; color: #fff; }
.btn-secondary { background: #f5f5f5; color: #555; border: 1px solid #ddd; }
.btn-danger { background: #333; color: #fff; }

.modal-overlay {
    display: none;
    position: fixed;
    inset: 0;
    background: rgba(0,0,0,0.5);
    z-index: 1000;
    justify-content: center;
    align-items: flex-start;
    padding: 40px 20px;
    overflow-y: auto;
}
.modal-overlay.show { display: flex; }

.modal {
    background: #fff;
    max-width: 700px;
    width: 100%;
    padding: 36px;
    position: relative;
    border-top: 3px solid #005c2e;
    box-shadow: 0 8px 32px rgba(0,0,0,0.12);
}

.modal-close {
    position: absolute;
    top: 14px;
    right: 18px;
    font-size: 20px;
    cursor: pointer;
    color: #bbb;
    border: none;
    background: none;
    line-height: 1;
}
.modal-close:hover { color: #1a1a1a; }

.modal-category {
    font-size: 11px;
    font-weight: 700;
    color: #005c2e;
    letter-spacing: 0.5px;
    margin-bottom: 10px;
}

.modal-title {
    font-size: 23px;
    font-weight: 800;
    line-height: 1.35;
    margin-bottom: 12px;
    letter-spacing: -0.5px;
    padding-bottom: 14px;
    border-bottom: 1px solid #eee;
}

.modal-meta {
    font-size: 12px;
    color: #aaa;
    margin-bottom: 20px;
    display: flex;
    gap: 14px;
}

.modal-content {
    font-size: 15px;
    line-height: 1.9;
    color: #333;
    white-space: pre-wrap;
    word-break: break-word;
}

.modal-actions {
    display: flex;
    gap: 8px;
    margin-top: 24px;
    padding-top: 14px;
    border-top: 1px solid #eee;
}
.modal-actions .btn { flex: none; padding: 9px 18px; }

.toast {
    position: fixed;
    bottom: 24px;
    right: 24px;
    min-width: 220px;
    padding: 11px 16px;
    background: #1a1a1a;
    color: #fff;
    font-size: 13px;
    font-weight: 600;
    border-left: 3px solid #005c2e;
    opacity: 0;
    transform: translateY(8px);
    pointer-events: none;
    transition: opacity 0.2s, transform 0.2s;
    z-index: 9999;
    border-radius: 2px;
}
.toast.show { opacity: 1; transform: translateY(0); }
.toast.error { border-left-color: #ff4444; }

.loading {
    grid-column: 1 / -1;
    text-align: center;
    padding: 60px;
    color: #ccc;
    font-size: 13px;
}

Overwriting /content/chungjung/static/style.css


In [94]:
%%writefile /content/chungjung/server.js

const express = require('express');
const path = require('path');
const mysql = require('mysql2/promise');

const app = express();
const PORT = 3000;

app.use(express.json());
app.use(express.urlencoded({ extended: true }));
app.use('/static', express.static(path.join(__dirname, 'static')));

const pool = mysql.createPool({
    host: 'localhost',
    user: 'testuser',
    password: '1234',
    database: 'testdb',
    waitForConnections: true,
    connectionLimit: 10,
    queueLimit: 0
});

async function checkDbConnection() {
    try {
        const conn = await pool.getConnection();
        console.log('DB 연결 성공');
        conn.release();
    } catch (err) {
        console.error('DB 연결 실패:', err);
    }
}
checkDbConnection();

app.get('/', (req, res) => {
    res.sendFile(path.join(__dirname, 'templates', 'index.html'));
});

app.get('/api/news', async (req, res) => {
    try {
        const [rows] = await pool.query(
            'SELECT id, title, category, author, created_at FROM news ORDER BY id DESC'
        );
        res.json({ success: true, data: rows });
    } catch (err) {
        console.error('목록 조회 실패:', err);
        res.status(500).json({ success: false, message: '기사 목록을 불러오는 데 실패했습니다.' });
    }
});

app.get('/api/news/:id', async (req, res) => {
    try {
        const [rows] = await pool.query(
            'SELECT * FROM news WHERE id = ?',
            [req.params.id]
        );
        if (rows.length === 0) {
            return res.status(404).json({ success: false, message: '기사를 찾을 수 없습니다.' });
        }
        res.json({ success: true, data: rows[0] });
    } catch (err) {
        console.error('단건 조회 실패:', err);
        res.status(500).json({ success: false, message: '기사 조회 중 오류가 발생했습니다.' });
    }
});

app.post('/api/news', async (req, res) => {
    try {
        const { title, category, content, author } = req.body;
        if (!title || !category || !content || !author) {
            return res.status(400).json({ success: false, message: '모든 항목을 입력해주세요.' });
        }
        await pool.query(
            'INSERT INTO news (title, category, content, author) VALUES (?, ?, ?, ?)',
            [title, category, content, author]
        );
        res.json({ success: true, message: '기사가 송고되었습니다.' });
    } catch (err) {
        console.error('기사 등록 실패:', err);
        res.status(500).json({ success: false, message: '기사 송고 중 오류가 발생했습니다.' });
    }
});

app.put('/api/news/:id', async (req, res) => {
    try {
        const { title, category, content, author } = req.body;
        const [result] = await pool.query(
            'UPDATE news SET title=?, category=?, content=?, author=? WHERE id=?',
            [title, category, content, author, req.params.id]
        );
        if (result.affectedRows === 0) {
            return res.status(404).json({ success: false, message: '기사를 찾을 수 없습니다.' });
        }
        res.json({ success: true, message: '기사가 수정되었습니다.' });
    } catch (err) {
        console.error('기사 수정 실패:', err);
        res.status(500).json({ success: false, message: '기사 수정 중 오류가 발생했습니다.' });
    }
});

app.delete('/api/news/:id', async (req, res) => {
    try {
        const [result] = await pool.query(
            'DELETE FROM news WHERE id=?',
            [req.params.id]
        );
        if (result.affectedRows === 0) {
            return res.status(404).json({ success: false, message: '기사를 찾을 수 없습니다.' });
        }
        res.json({ success: true, message: '기사가 삭제되었습니다.' });
    } catch (err) {
        console.error('기사 삭제 실패:', err);
        res.status(500).json({ success: false, message: '기사 삭제 중 오류가 발생했습니다.' });
    }
});

app.get('/write', (req, res) => {
    res.sendFile(path.join(__dirname, 'templates', 'write.html'));
});

app.get('/edit/:id', (req, res) => {
    res.sendFile(path.join(__dirname, 'templates', 'edit.html'));
});

app.listen(PORT, () => {
    console.log('========================================');
    console.log(` 충정일보 서버 실행 중: http://localhost:${PORT}`);
    console.log('========================================');
});

Overwriting /content/chungjung/server.js


In [95]:
%%writefile /content/chungjung/templates/index.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>충정일보</title>
    <link rel="stylesheet" href="/static/style.css">
</head>
<body>

<header class="newspaper-header">
    <div class="header-inner">
        <div class="header-logo">
            <img src="https://raw.githubusercontent.com/flathfk/Bootcamp_Study/main/chungjung_news_logo.png" alt="충정일보" style="height:100px;">
        </div>
        <div class="header-top">
            <div class="header-meta-right">
                <div id="today-date"></div>
                <div>대한민국 대표 디지털 신문 · 제 1호</div>
            </div>
        </div>
        <nav class="header-nav">
            <a href="/" class="active">전체</a>
            <a href="/write">기사 송고</a>
        </nav>
    </div>
</header>

<div class="page">
    <div class="section-header">
        <span class="section-label">최신 기사</span>
        <span class="section-date" id="today-date-sub"></span>
    </div>

    <!-- 메인 그리드 (헤드라인 + 사이드) -->
    <div class="news-grid" id="mainGrid">
        <div class="loading">기사를 불러오는 중...</div>
    </div>

    <!-- 하단 일반 기사 그리드 -->
    <div class="news-grid-bottom" id="bottomGrid" style="display:none;"></div>
</div>

<!-- 기사 상세 모달 -->
<div class="modal-overlay" id="modalOverlay">
    <div class="modal">
        <button class="modal-close" onclick="closeModal()">✕</button>
        <div class="modal-category" id="modalCategory"></div>
        <h2 class="modal-title" id="modalTitle"></h2>
        <div class="modal-meta">
            <span id="modalAuthor"></span>
            <span id="modalDate"></span>
        </div>
        <div class="modal-content" id="modalContent"></div>
        <div class="modal-actions">
            <button class="btn btn-secondary" onclick="closeModal()">닫기</button>
            <button class="btn btn-primary" id="modalEditBtn">수정</button>
            <button class="btn btn-danger" id="modalDeleteBtn">삭제</button>
        </div>
    </div>
</div>

<div class="toast" id="toast"></div>

<script>
    // 오늘 날짜
    document.getElementById('today-date').textContent =
        new Date().toLocaleDateString('ko-KR', { year:'numeric', month:'long', day:'numeric', weekday:'long' });

    // 토스트
    function showToast(message, isError = false) {
        const toast = document.getElementById('toast');
        toast.textContent = message;
        toast.className = isError ? 'toast show error' : 'toast show';
        setTimeout(() => { toast.className = isError ? 'toast error' : 'toast'; }, 2500);
    }

    // 기사 목록 로드
    async function loadNews() {
        try {
            const res = await fetch('/api/news');
            const result = await res.json();

            if (!result.success) throw new Error(result.message);

            renderNews(result.data);
        } catch (err) {
            document.getElementById('mainGrid').innerHTML =
                '<div class="empty-state">기사를 불러오지 못했습니다.</div>';
        }
    }

    function escapeHtml(str) {
        if (!str) return '';
        return String(str)
            .replace(/&/g, '&amp;')
            .replace(/</g, '&lt;')
            .replace(/>/g, '&gt;')
            .replace(/"/g, '&quot;');
    }

    function formatDate(str) {
        if (!str) return '';
        return new Date(str).toLocaleDateString('ko-KR', { month:'long', day:'numeric', hour:'2-digit', minute:'2-digit' });
    }

    function renderNews(articles) {
        const mainGrid = document.getElementById('mainGrid');
        const bottomGrid = document.getElementById('bottomGrid');

        if (articles.length === 0) {
            mainGrid.innerHTML = '<div class="empty-state">아직 등록된 기사가 없습니다.<br>첫 번째 기사를 송고해보세요.</div>';
            bottomGrid.style.display = 'none';
            return;
        }

        // 헤드라인: 첫 번째 기사
        const headline = articles[0];
        let mainHtml = `
            <div class="card-headline" onclick="openModal(${headline.id})">
                <div class="card-category">${escapeHtml(headline.category)}</div>
                <hr class="card-divider">
                <div class="card-title">${escapeHtml(headline.title)}</div>
                <div class="card-content">${escapeHtml(headline.content)}</div>
                <div class="card-meta">
                    <span>기자 ${escapeHtml(headline.author)}</span>
                    <span>${formatDate(headline.created_at)}</span>
                </div>
            </div>
        `;

        // 사이드바: 2~4번째 기사 (최대 3개)
        const sideArticles = articles.slice(1, 4);
        if (sideArticles.length > 0) {
            let sideHtml = '<div class="news-sidebar">';
            sideArticles.forEach(a => {
                sideHtml += `
                    <div class="card-side" onclick="openModal(${a.id})">
                        <div class="card-category">${escapeHtml(a.category)}</div>
                        <div class="card-title">${escapeHtml(a.title)}</div>
                        <div class="card-content">${escapeHtml(a.content)}</div>
                        <div class="card-meta">
                            <span>${escapeHtml(a.author)}</span>
                            <span>${formatDate(a.created_at)}</span>
                        </div>
                    </div>
                `;
            });
            sideHtml += '</div>';
            mainHtml += sideHtml;
        }

        mainGrid.innerHTML = mainHtml;

        // 하단 그리드: 5번째 이후 기사
        const bottomArticles = articles.slice(4);
        if (bottomArticles.length > 0) {
            bottomGrid.style.display = 'grid';
            bottomGrid.innerHTML = bottomArticles.map(a => `
                <div class="card-normal" onclick="openModal(${a.id})">
                    <div class="card-category">${escapeHtml(a.category)}</div>
                    <div class="card-title" style="font-size:17px;">${escapeHtml(a.title)}</div>
                    <div class="card-content">${escapeHtml(a.content)}</div>
                    <div class="card-meta">
                        <span>${escapeHtml(a.author)}</span>
                        <span>${formatDate(a.created_at)}</span>
                    </div>
                </div>
            `).join('');
        } else {
            bottomGrid.style.display = 'none';
        }
    }

    // 모달 열기
    let currentArticleId = null;

    async function openModal(id) {
        try {
            const res = await fetch(`/api/news/${id}`);
            const result = await res.json();
            if (!result.success) throw new Error(result.message);

            const a = result.data;
            currentArticleId = id;

            document.getElementById('modalCategory').textContent = a.category;
            document.getElementById('modalTitle').textContent = a.title;
            document.getElementById('modalAuthor').textContent = `기자 ${a.author}`;
            document.getElementById('modalDate').textContent = formatDate(a.created_at);
            document.getElementById('modalContent').textContent = a.content;
            document.getElementById('modalEditBtn').onclick = () => { window.location.href = `/edit/${id}`; };
            document.getElementById('modalDeleteBtn').onclick = () => deleteArticle(id);

            document.getElementById('modalOverlay').classList.add('show');
        } catch (err) {
            showToast('기사를 불러오지 못했습니다.', true);
        }
    }

    function closeModal() {
        document.getElementById('modalOverlay').classList.remove('show');
        currentArticleId = null;
    }

    // 모달 외부 클릭 시 닫기
    document.getElementById('modalOverlay').addEventListener('click', function(e) {
        if (e.target === this) closeModal();
    });

    // 삭제
    async function deleteArticle(id) {
        if (!confirm('이 기사를 삭제하시겠습니까?')) return;

        try {
            const res = await fetch(`/api/news/${id}`, { method: 'DELETE' });
            const result = await res.json();

            if (!result.success) throw new Error(result.message);

            closeModal();
            showToast(result.message);
            await loadNews();
        } catch (err) {
            showToast(err.message, true);
        }
    }

    // 초기 로드
    loadNews();
</script>
</body>
</html>

Overwriting /content/chungjung/templates/index.html


In [96]:
%%writefile /content/chungjung/templates/write.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>기사 송고 - 충정일보</title>
    <link rel="stylesheet" href="/static/style.css">
</head>
<body>

<header class="newspaper-header">
    <div class="header-inner">
        <div class="header-logo">
            <img src="https://raw.githubusercontent.com/flathfk/Bootcamp_Study/main/chungjung_news_logo.png" alt="충정일보" style="height:100px;">
        </div>
        <div class="header-top">
            <div class="header-meta-right">
                <div>대한민국 대표 디지털 신문 · 제 1호</div>
            </div>
        </div>
        <nav class="header-nav">
            <a href="/">전체</a>
            <a href="/write" class="active">기사 송고</a>
        </nav>
    </div>
</header>

<div class="form-page">
    <div class="form-header">
        <h1>기사 송고</h1>
        <p>작성한 기사를 충정일보 데스크로 송고합니다.</p>
    </div>

    <div class="form-panel">
        <form id="writeForm">
            <div class="form-group">
                <label>제목</label>
                <input type="text" name="title" placeholder="기사 제목을 입력하세요" required>
            </div>

            <div class="form-group">
                <label>카테고리</label>
                <select name="category" required>
                    <option value="">카테고리 선택</option>
                    <option value="정치">정치</option>
                    <option value="경제">경제</option>
                    <option value="사회">사회</option>
                    <option value="국제">국제</option>
                    <option value="연예">연예</option>
                    <option value="스포츠">스포츠</option>
                    <option value="IT">IT</option>
                    <option value="문화">문화</option>
                </select>
            </div>

            <div class="form-group">
                <label>기자명</label>
                <input type="text" name="author" placeholder="기자 이름" required>
            </div>

            <div class="form-group">
                <label>본문</label>
                <textarea name="content" placeholder="기사 본문을 입력하세요..." required></textarea>
            </div>

            <div class="button-row">
                <button class="btn btn-primary" type="submit">송고하기</button>
                <a class="btn btn-secondary" href="/" style="text-align:center; display:flex; align-items:center; justify-content:center;">취소</a>
            </div>
        </form>
    </div>
</div>

<div class="toast" id="toast"></div>

<script>
    function showToast(message, isError = false) {
        const toast = document.getElementById('toast');
        toast.textContent = message;
        toast.className = isError ? 'toast show error' : 'toast show';
        setTimeout(() => { toast.className = isError ? 'toast error' : 'toast'; }, 2500);
    }

    document.getElementById('writeForm').addEventListener('submit', async function(e) {
        e.preventDefault();

        const formData = new FormData(this);
        const payload = Object.fromEntries(formData.entries());

        try {
            const res = await fetch('/api/news', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify(payload)
            });

            const result = await res.json();

            if (!result.success) throw new Error(result.message);

            showToast(result.message);
            setTimeout(() => { window.location.href = '/'; }, 800);
        } catch (err) {
            showToast(err.message, true);
        }
    });
</script>
</body>
</html>

Overwriting /content/chungjung/templates/write.html


In [97]:
%%writefile /content/chungjung/templates/edit.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>기사 수정 - 충정일보</title>
    <link rel="stylesheet" href="/static/style.css">
</head>
<body>

<header class="newspaper-header">
    <div class="header-inner">
        <div class="header-logo">
            <img src="https://raw.githubusercontent.com/flathfk/Bootcamp_Study/main/chungjung_news_logo.png" alt="충정일보" style="height:100px;">
        </div>
        <div class="header-top">
            <div class="header-meta-right">
                <div>대한민국 대표 디지털 신문 · 제 1호</div>
            </div>
        </div>
        <nav class="header-nav">
            <a href="/">전체</a>
            <a href="/write">기사 송고</a>
        </nav>
    </div>
</header>

<div class="form-page">
    <div class="form-header">
        <h1>기사 수정</h1>
        <p>기사 내용을 수정합니다.</p>
    </div>

    <div class="form-panel">
        <form id="editForm">
            <div class="form-group">
                <label>제목</label>
                <input type="text" id="title" name="title" required>
            </div>

            <div class="form-group">
                <label>카테고리</label>
                <select id="category" name="category" required>
                    <option value="정치">정치</option>
                    <option value="경제">경제</option>
                    <option value="사회">사회</option>
                    <option value="국제">국제</option>
                    <option value="연예">연예</option>
                    <option value="스포츠">스포츠</option>
                    <option value="IT">IT</option>
                    <option value="문화">문화</option>
                </select>
            </div>

            <div class="form-group">
                <label>기자명</label>
                <input type="text" id="author" name="author" required>
            </div>

            <div class="form-group">
                <label>본문</label>
                <textarea id="content" name="content" required></textarea>
            </div>

            <div class="button-row">
                <button class="btn btn-primary" type="submit">수정 완료</button>
                <a class="btn btn-secondary" href="/" style="text-align:center; display:flex; align-items:center; justify-content:center;">취소</a>
            </div>
        </form>
    </div>
</div>

<div class="toast" id="toast"></div>

<script>
    // URL에서 id 추출: /edit/3 → 3
    const articleId = window.location.pathname.split('/').pop();

    function showToast(message, isError = false) {
        const toast = document.getElementById('toast');
        toast.textContent = message;
        toast.className = isError ? 'toast show error' : 'toast show';
        setTimeout(() => { toast.className = isError ? 'toast error' : 'toast'; }, 2500);
    }

    // 기존 데이터 불러오기
    async function loadArticle() {
        try {
            const res = await fetch(`/api/news/${articleId}`);
            const result = await res.json();

            if (!result.success) throw new Error(result.message);

            const a = result.data;
            document.getElementById('title').value = a.title;
            document.getElementById('author').value = a.author;
            document.getElementById('content').value = a.content;

            // 카테고리 select 현재값 선택
            const categorySelect = document.getElementById('category');
            for (const opt of categorySelect.options) {
                if (opt.value === a.category) {
                    opt.selected = true;
                    break;
                }
            }
        } catch (err) {
            showToast('기사 정보를 불러오지 못했습니다.', true);
        }
    }

    // 수정 제출
    document.getElementById('editForm').addEventListener('submit', async function(e) {
        e.preventDefault();

        const formData = new FormData(this);
        const payload = Object.fromEntries(formData.entries());

        try {
            const res = await fetch(`/api/news/${articleId}`, {
                method: 'PUT',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify(payload)
            });

            const result = await res.json();

            if (!result.success) throw new Error(result.message);

            showToast(result.message);
            setTimeout(() => { window.location.href = '/'; }, 800);
        } catch (err) {
            showToast(err.message, true);
        }
    });

    loadArticle();
</script>
</body>
</html>

Overwriting /content/chungjung/templates/edit.html




---



In [64]:
%%bash
cd /content/chungjung && npm install


up to date, audited 81 packages in 770ms

18 packages are looking for funding
  run `npm fund` for details

found 0 vulnerabilities




---



In [65]:
%%bash
mysql -u root -e "
CREATE USER IF NOT EXISTS 'testuser'@'localhost' IDENTIFIED BY '1234';
GRANT ALL PRIVILEGES ON testdb.* TO 'testuser'@'localhost';
FLUSH PRIVILEGES;
"

mysql -u testuser -p1234 testdb -e "
CREATE TABLE IF NOT EXISTS news (
    id INT AUTO_INCREMENT PRIMARY KEY,
    title VARCHAR(300) NOT NULL,
    category VARCHAR(50) NOT NULL,
    content TEXT NOT NULL,
    author VARCHAR(100) NOT NULL,
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP
);
"



---



In [66]:
%%bash
cd /content/project
npm install express mysql2 bcrypt
npm install -g cloudflared


up to date, audited 80 packages in 1s

24 packages are looking for funding
  run `npm fund` for details

found 0 vulnerabilities

changed 1 package in 2s


bash: line 1: cd: /content/project: No such file or directory


In [67]:
%%bash
cd /content/chungjung
nohup node server.js > /content/server.log 2>&1 &
sleep 2
cat /content/server.log

 충정일보 서버 실행 중: http://localhost:3000
DB 연결 성공


In [68]:
%%bash
nohup cloudflared tunnel --url http://localhost:3000 > /content/tunnel.log 2>&1 &
sleep 5
cat /content/tunnel.log | grep trycloudflare.com

2026-03-19T06:41:55Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-19T06:41:59Z INF |  https://fruits-combination-kevin-flow.trycloudflare.com                                   |




---



In [71]:
%%bash
cat /content/tunnel.log | grep trycloudflare.com

2026-03-19T06:41:55Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-19T06:41:59Z INF |  https://fruits-combination-kevin-flow.trycloudflare.com                                   |
